# CSE 242 — Assignment 2
**Name:** Siddhesh Kukade  
**Student ID:** 2346622

This notebook contains solutions for Questions 1–3. Run the cells from top to bottom.

**Note:** Place `Advertising.csv` in the same folder as this notebook before running Question 2. The assignment handout names this dataset but it was not included with the uploaded file. The notebook checks for it and reports a clear error if it is missing.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits

%matplotlib inline


## Question 1 — Maximum Likelihood Estimation and Posterior (20 pts)

Let \\(p\\) be the probability of heads. We observe 3 heads and 2 tails in 5 independent flips.

### (a) Maximum likelihood estimate

The likelihood is
\\[
L(p)=\binom{5}{3}p^3(1-p)^2.
\\]
The log-likelihood is
\\[
\ell(p)=\log\binom{5}{3}+3\log p+2\log(1-p).
\\]
Differentiating and setting the derivative to zero:
\\[
\frac{d\ell}{dp}=\frac{3}{p}-\frac{2}{1-p}=0
\quad\Longrightarrow\quad 3(1-p)=2p.
\\]
Therefore,
\\[
\boxed{\hat p_{MLE}=\frac{3}{5}=0.6.}
\\]

### (b) Posterior density

Using Bayes' rule, for prior density \\(f(p)\\) on \\(0\leq p\leq1\\),
\\[
\boxed{
f(p\mid D)=
\frac{p^3(1-p)^2f(p)}
{\displaystyle\int_0^1 u^3(1-u)^2f(u)\,du}
}
\qquad 0\leq p\leq1.
\\]
The binomial coefficient cancels between the numerator and normalizing denominator.

## Question 2 — Linear Regression (40 pts)

The target is `Sales`; the predictors are `TV`, `Radio`, and `Newspaper`.

In [ ]:
# Read the Advertising dataset (keep Advertising.csv beside this notebook).
from pathlib import Path

csv_path = Path("Advertising.csv")
if not csv_path.exists():
    raise FileNotFoundError(
        "Advertising.csv was not found. Download the Advertising.csv dataset "
        "specified in the assignment and place it in the same folder as this notebook."
    )

data = pd.read_csv(csv_path, index_col=0)
display(data.head())
print("Dataset shape:", data.shape)


In [ ]:
# Prepare features and response
X = data[['TV', 'Radio', 'Newspaper']]
y = data['Sales']

print(type(X), X.shape)
print(type(y), y.shape)


### Question 2.1 — Train/test split (5 pts)

Use an 80% training / 20% testing split. A fixed random state makes the results reproducible and ensures the same rows are used in the comparison in Question 2.5.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test: ", X_test.shape)
print("y_test: ", y_test.shape)


### Question 2.2 — Train the linear regression model (10 pts)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Intercept:", round(model.intercept_, 4))
print("Coefficients:", [round(v, 4) for v in model.coef_])
print("Feature coefficients:", dict(zip(X.columns, model.coef_)))


### Question 2.3 — Interpret the coefficients (10 pts)

The fitted model is printed below using the coefficients learned from the supplied dataset and this notebook's fixed split.

In [ ]:
coef_names = list(X.columns)
equation = f"Sales = {model.intercept_:.4f}"
for name, coef in zip(coef_names, model.coef_):
    sign = "+" if coef >= 0 else "-"
    equation += f" {sign} {abs(coef):.4f}({name})"
print(equation)
print(
    f"TV interpretation: holding Radio and Newspaper spending constant, "
    f"an additional $1,000 spent on TV advertising is associated with an "
    f"estimated change of {model.coef_[0]:.4f} thousand units in Sales."
)


**Interpretation:** The TV coefficient represents the estimated change in sales (in thousands of items) associated with a one-unit increase in TV advertising (one thousand dollars), while holding Radio and Newspaper advertising constant. A positive coefficient indicates an increase in predicted sales; a negative coefficient indicates a decrease. This is an association in the fitted linear model, not necessarily a causal effect.

### Make predictions and evaluate the model

In [ ]:
y_pred = model.predict(X_test)
print("R-squared on test data:", model.score(X_test, y_test))


### Question 2.4 — Test RMSE (5 pts)

RMSE is the square root of the mean squared prediction error.

In [ ]:
mse = metrics.mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print("MSE:", mse)
print("RMSE:", rmse)


### Question 2.5 — Does Newspaper help? (10 pts)

Repeat the same split with only `TV` and `Radio`. The same `random_state` and test size ensure the comparison uses the same train/test observations.

In [ ]:
X_two = data[['TV', 'Radio']]
X_two_train, X_two_test, y_two_train, y_two_test = train_test_split(
    X_two, y, test_size=0.20, random_state=42
)

model_two = LinearRegression()
model_two.fit(X_two_train, y_two_train)
y_two_pred = model_two.predict(X_two_test)

rmse_two = np.sqrt(metrics.mean_squared_error(y_two_test, y_two_pred))

print("RMSE with TV, Radio, Newspaper:", rmse)
print("RMSE with TV and Radio only:   ", rmse_two)
print("Difference (three-feature minus two-feature):", rmse - rmse_two)


**Observations:** Compare the two test RMSE values. A lower RMSE means better prediction on this test set. If the model using only TV and Radio has a lower RMSE, Newspaper does not improve this particular test-set prediction and may add noise. If the three-feature model has a lower RMSE, Newspaper provides some additional predictive information. The numerical comparison above determines the conclusion for this split.

## Question 3 — Principal Component Analysis (40 pts)

We use the 8×8 handwritten digits dataset from scikit-learn. Each image has 64 pixel features.

In [ ]:
# Load the digits dataset
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

data_digit = load_digits()
X_digit = data_digit.data
print("Digit feature matrix shape:", X_digit.shape)


In [ ]:
# Helper function supplied/adapted from the assignment handout
def display_digits(imgs):
    h, w = 3, 5
    fig, axes = plt.subplots(h, w, figsize=(10, 6))
    for i in range(h):
        for j in range(w):
            idx = i * w + j
            if idx >= len(imgs):
                axes[i, j].axis('off')
                continue
            if imgs.shape[1] == 64:
                axes[i, j].imshow(imgs[idx].reshape(8, 8), cmap='gray')
            else:
                axes[i, j].imshow(imgs[idx], cmap='gray')
            axes[i, j].axis('off')
            axes[i, j].set_title(f"label {data_digit.target[idx]}")
    plt.tight_layout()
    plt.show()

display_digits(data_digit.images)


### Question 3.1 — Reduce to two dimensions (10 pts)

Fit PCA with two principal components and transform the 64-dimensional feature vectors.

In [ ]:
pca_2 = PCA(n_components=2)
projected = pca_2.fit_transform(X_digit)

print("Original shape:", X_digit.shape)
print("Projected shape:", projected.shape)


In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(
    projected[:, 0], projected[:, 1],
    c=data_digit.target, edgecolor='none', alpha=0.7,
    cmap=plt.cm.get_cmap('Paired', 10)
)
plt.xlabel('1st Component')
plt.ylabel('2nd Component')
plt.colorbar(label='Digit label')
plt.title('Digits projected onto the first two principal components')
plt.show()


### Question 3.2 — Explained variance (15 pts)

Fit PCA with all components, print the explained variance ratio for each component, and plot the ratios.

In [ ]:
pca_all = PCA()
pca_all.fit(X_digit)

explained_ratios = pca_all.explained_variance_ratio_
print("Explained variance ratio for each component:")
for i, ratio in enumerate(explained_ratios, start=1):
    print(f"Component {i}: {ratio:.6f}")

plt.figure(figsize=(10, 5))
plt.plot(
    np.arange(1, len(explained_ratios) + 1),
    explained_ratios, marker='o'
)
plt.xlabel('The $i-$th component')
plt.ylabel('Ratio of explained variance')
plt.title('Explained variance ratio by principal component')
plt.xticks(np.arange(0, 65, 5))
plt.grid(True)
plt.show()

print("Cumulative explained variance (first 10 components):")
print(np.round(np.cumsum(explained_ratios)[:10], 4))


### Question 3.3 — Observations from eigenvectors (5 pts)

Each eigenvector (principal component) is a 64-dimensional direction that can be reshaped into an 8×8 image. The first components generally capture broad, high-variance patterns in the digit images, such as strokes, edges, and overall shape. Later components capture progressively finer variations and details. The eigenvector images can contain both positive and negative pixel weights; they are directions of variation rather than ordinary digit images.

In [ ]:
# Visualize all 64 eigenvectors as 8x8 images
fig, axes = plt.subplots(6, 12, figsize=(14, 8))
for i, ax in enumerate(axes.flat):
    if i >= 64:
        ax.axis('off')
    else:
        ax.imshow(pca_all.components_[i].reshape(8, 8), cmap='gray')
        ax.axis('off')
        ax.set_title(f'{i + 1}')
fig.suptitle('Eigenvectors (principal components) of the digit images')
plt.tight_layout()
plt.show()


### Feature reconstruction using 80% explained variance

In [ ]:
pca_08 = PCA(n_components=0.8)
lower_dimension = pca_08.fit_transform(X_digit)
reconstructed = pca_08.inverse_transform(lower_dimension)

print("Number of components retained:", pca_08.n_components_)
print("Total explained variance retained:", pca_08.explained_variance_ratio_.sum())
display_digits(reconstructed)


### Question 3.4 — Does PCA mitigate random noise? (10 pts)

Add random Gaussian noise, fit PCA retaining 80% of the variance, and reconstruct the noisy images.

In [ ]:
# Add random noise; fixed seed makes the experiment reproducible
rng = np.random.default_rng(42)
noisy_imgs = rng.normal(loc=X_digit, scale=5, size=X_digit.shape)

display_digits(noisy_imgs)


In [ ]:
# Fit PCA on noisy data while retaining 80% of its variance
pca_noise = PCA(n_components=0.8)
noise_low_dimension = pca_noise.fit_transform(noisy_imgs)
reconstructed_noise = pca_noise.inverse_transform(noise_low_dimension)

print("Number of components retained:", pca_noise.n_components_)
print("Explained variance retained:", pca_noise.explained_variance_ratio_.sum())
display_digits(reconstructed_noise)


**Written answer:** PCA can mitigate some random noise because it represents the data using a smaller number of principal components that capture the dominant patterns of variation. Random pixel-level noise often contributes to lower-variance directions, which may be discarded when the data is projected onto the retained components. Reconstructing from those components can therefore produce images that look cleaner than the noisy inputs. However, PCA does not remove all noise: some noise may lie in retained directions, and useful fine details may also be lost. The result depends on the variance threshold and the noise level.

## Acknowledgment

The assignment instructions require disclosure of help from generative AI. If you submit this notebook, acknowledge the assistance you used in accordance with your course policy. Review and understand each answer, and verify the results by running the notebook with the course-provided `Advertising.csv` dataset.